# Exploring the limits of eagerly allocated arrays

In this demo, we will explore how the limits of our memory by (eagerly) allocating `numpy` arrays of increasing size.

**Note**: Implementations of memory handling at the operating system level are complex and vary across Mac, Linux and Windows. Here, we gloss over this complexity by limiting the memory available to this notebook artificially to a fraction of the available physical memory. This allows us to harmonise the operating systems for teaching purposes, and to stop Python crashing our computer by catching `MemoryErrors`. 


In the `psutil` and `sys` modules Python has some built-in functionality to get information about the hardware and operating system, respectively. We can print some of this information to get an idea:


In [ ]:
import sys

import psutil

vm = psutil.virtual_memory()
print(f"Platform: {sys.platform}")
print(f"Total RAM: {vm.total / 1024**3:.2f} GiB")
print(f"Available RAM: {vm.available / 1024**3:.2f} GiB")

We can use a helper function to limit this notebook's available memory to 50% of the phyical memory. 


In [ ]:
from course_large_array_data.cap_memory import enable_process_memory_limit

cap_bytes = enable_process_memory_limit(fraction=0.2)
print(f"Applied cap: {cap_bytes / 1024**3:.2f} GiB")

This then allows us to demonstrate that allocating an array that is larger than this causes an error (whilst avoiding a crash at the operating system level).


In [ ]:
import numpy as np

target_gib = vm.total / 1024**3 / 4  # a quarter the total physical memory
n_elements = int(target_gib * 1024**3 / np.dtype(np.float64).itemsize)
print(f"Target array size: {target_gib:.2f} GiB")

try:
    arr = np.random.random(n_elements).astype(np.float64)
    print(f"Unexpected success: {arr.nbytes / 1024**3:.2f} GiB")
except (MemoryError, OSError) as exc:
    print(f"Allocation failed as expected: {type(exc).__name__}: {exc}")

We can remove our memory cap with the opposite helper function.

In [ ]:
from course_large_array_data.cap_memory import disable_process_memory_limit

disable_process_memory_limit()
print("Memory cap removed.")

With our memory limit remove, we can now allocate the array that we couldn't allocate before.

In [ ]:
import numpy as np

target_gib = vm.total / 1024**3 / 4  # a quarter of the total physical memory
n_elements = int(target_gib * 1024**3 / np.dtype(np.float64).itemsize)
print(f"Target array size: {target_gib:.2f} GiB")

try:
    arr = np.random.random(n_elements).astype(np.float64)
    print(
        f"Array allocated successfully with limits removed: {arr.nbytes / 1024**3:.2f} GiB"
    )
except (MemoryError, OSError) as exc:
    print(f"Allocation failed: {type(exc).__name__}: {exc}")

Note that because we've removed our self-imposed memory restriction, we run the danger of crashing our operating system if we allocate too large arrays eagerly.

We have commented out the dangerous code in the cell below, but you are welcome to uncomment it and try running it if you are feeling brave (you may have to restart your computer with the power button though!).

In [ ]:
import numpy as np
import psutil

vm = psutil.virtual_memory()

target_gib = vm.total / 1024**3 / 4
for i in range(5):
    n_elements = int(target_gib * 1024**3 / np.dtype(np.float64).itemsize)
    print(f"Target array size: {target_gib:.2f} GiB")

    # Fair warning: the below will likely crash your operating system.
    # And Python cannot catch it.
    # Try at your own peril (save your work before, and restart your computer with the on/off button after.).

    # import time
    # try:
    #     t0 = time.perf_counter()
    #     arr = np.random.random(n_elements).astype(np.float64)
    #     elapsed_s = time.perf_counter() - t0
    #     print(f"Allocation of {arr.nbytes / 1024**3:.2f} GiB successful in {elapsed_s:.3f} s")
    # except Exception as exc:
    #     print(f"Crash unexpectedly caught by Python: {type(exc).__name__}: {exc}")

    target_gib *= 2

## Stretch exercise

Slowly increase the self-imposed memory limit and allocate larger and larger arrays in the cell below. Can you allocate arrays larger than your physical memory? Why (not)?

Alternatively, feel free to think about the size of the data you typically analyse, and how it relates to the physical memory available on your laptop. You could also invent your own related stretch exercise or help others in the room.

In [ ]:
enable_process_memory_limit(fraction=0.2)  # modify
target_gib = vm.total / 1024**3 / 4  # modify

n_elements = int(target_gib * 1024**3 / np.dtype(np.float64).itemsize)
print(f"Target array size: {target_gib:.2f} GiB")

try:
    arr = np.random.random(n_elements).astype(np.float64)
    print(f"Array allocated successfully: {arr.nbytes / 1024**3:.2f} GiB")
except (MemoryError, OSError) as exc:
    print(f"Allocation failed: {type(exc).__name__}: {exc}")